In [ ]:
!pip -q install yfinance optuna scikit-learn joblib pandas numpy matplotlib

import os, json, time, itertools, hashlib, random
from copy import deepcopy
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
import optuna
from optuna.samplers import TPESampler

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, balanced_accuracy_score,
    accuracy_score, confusion_matrix, classification_report, precision_recall_curve
)

import joblib
import matplotlib.pyplot as plt

np.set_printoptions(suppress=True)
pd.set_option("display.max_columns", None)
print("Setup complete.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 7.8 MB/s eta 0:00:00
Setup complete.


In [ ]:
# ==== EDIT THESE ====
TICKER      = "NVDA"                 # e.g., AAPL, MSFT, GOOGL, META, AMZN, TSLA, NVDA
START       = "2015-01-01"
END         = "2025-08-01"

# Chronological split
TRAIN_END   = "2020-12-31"
VALID_END   = "2023-12-31"           # Test is (VALID_END, END]

STUDENT_ID  = "311438"
SEED_INDEX  = 0                      # use ONLY ONE seed this week: 0..4

# Metric for validation selection
METRIC      = "roc_auc"              # "roc_auc" | "average_precision" | "f1"
N_TRIALS    = 120                    # Optuna trials upper bound
OUTPUT_DIR  = "outputs_compare"      # artifacts will be stored here

Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)
print("OK: config loaded.")

OK: config loaded.


In [ ]:
def derive_seeds_from_student_id(student_id: str, n_seeds: int = 5):
    base = hashlib.sha256(student_id.strip().encode("utf-8")).hexdigest()
    seeds = []
    for i in range(n_seeds):
        chunk = base[i*8:(i+1)*8]  # 8 hex chars -> 32-bit int
        if len(chunk) < 8:
            chunk = (chunk + base)[:8]
        seeds.append(int(chunk, 16))
    return seeds

def download_close_series(ticker: str, start: str, end: str) -> pd.DataFrame:
    data = yf.download(ticker, start=start, end=end, interval="1d", auto_adjust=False, progress=False)
    if data.empty or "Close" not in data.columns:
        raise ValueError(f"No data/Close for {ticker}.")
    df = data[["Close"]].copy().reset_index()
    if "Date" not in df.columns and "Datetime" in df.columns:
        df = df.rename(columns={"Datetime":"Date"})
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    return df

def derive_labels_from_close(df: pd.DataFrame, q: float = 0.75):
    """Label day t as high-vol (1) if |log_ret_t| >= q-quantile across sample."""
    df = df.copy()
    df["log_ret"] = np.log(df["Close"]).diff()
    df = df.iloc[1:].reset_index(drop=True)  # drop first NaN
    abs_ret = df["log_ret"].abs().values
    thresh = np.quantile(abs_ret[~np.isnan(abs_ret)], q)
    y = (abs_ret >= thresh).astype(int)
    return df, y

def build_close_only_features(df_with_logret: pd.DataFrame) -> pd.DataFrame:
    """
    Close-only features. SHIFT by 1 day so features at t use info up to t-1 (no leakage).
    """
    close = df_with_logret["Close"]
    lr    = df_with_logret["log_ret"]

    feats = pd.DataFrame(index=df_with_logret.index)

    # Momentum & returns
    feats["ret1"]      = lr
    feats["ret2"]      = lr.shift(1)
    feats["ret3"]      = lr.shift(2)
    feats["ret5"]      = np.log(close).diff(5)
    feats["ret5_lag1"] = feats["ret5"].shift(1)

    # Moving averages (levels) & slopes
    feats["ma5"]       = close.rolling(5).mean()
    feats["ma10"]      = close.rolling(10).mean()
    feats["ma20"]      = close.rolling(20).mean()
    feats["ma_slope5"] = feats["ma5"].diff()
    feats["ma_slope10"]= feats["ma10"].diff()

    # Volatility features (variance/realized vol)
    feats["std5"]      = close.rolling(5).std()
    feats["std10"]     = close.rolling(10).std()
    feats["var10_ret"] = lr.rolling(10).var()
    feats["var20_ret"] = lr.rolling(20).var()
    feats["rv10"]      = (lr.pow(2)).rolling(10).sum()

    # Return of moving averages
    feats["ma5_ret"]   = np.log(feats["ma5"]).diff()
    feats["ma10_ret"]  = np.log(feats["ma10"]).diff()

    # SHIFT all by 1 day to avoid leakage
    feats = feats.shift(1)

    feats = feats.replace([np.inf, -np.inf], np.nan).dropna().copy()
    return feats

print("Helpers ready.")


Helpers ready.


In [ ]:
from sklearn.metrics import precision_recall_curve

def pick_threshold_by_metric(y_true, prob, mode="f1"):
    """Tune threshold on VALID using PR thresholds (good for imbalance)."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, prob)
    thresholds = np.append(thresholds, 1.0)  # align lengths
    best_thr, best_score = 0.5, -1.0
    for thr in thresholds:
        pred = (prob >= thr).astype(int)
        if mode == "f1":
            score = f1_score(y_true, pred, zero_division=0)
        else:
            score = balanced_accuracy_score(y_true, pred)
        if score > best_score:
            best_score, best_thr = score, thr
    return float(best_thr), float(best_score)

def score_on_valid(clf, X_train_s, y_train, X_valid_s, y_valid, metric="roc_auc"):
    clf.fit(X_train_s, y_train)
    prob = clf.predict_proba(X_valid_s)[:, 1]
    if metric == "roc_auc":
        return float(roc_auc_score(y_valid, prob))
    elif metric == "average_precision":
        return float(average_precision_score(y_valid, prob))
    else:
        thr, _ = pick_threshold_by_metric(y_valid, prob, mode="f1")
        pred = (prob >= thr).astype(int)
        return float(f1_score(y_valid, pred, zero_division=0))

def evaluate_on_test(best_params, X_train_s, y_train, X_valid_s, y_valid, X_test_s, y_test):
    """Threshold tuned on VALID (model trained on TRAIN), then final model trained on TRAIN+VALID -> TEST."""
    # Threshold selection on VALID (fit on TRAIN only)
    tmp = RandomForestClassifier(**best_params).fit(X_train_s, y_train)
    prob_valid = tmp.predict_proba(X_valid_s)[:, 1]
    thr_f1, _   = pick_threshold_by_metric(y_valid, prob_valid, mode="f1")
    thr_bal, _  = pick_threshold_by_metric(y_valid, prob_valid, mode="balanced")

    # Retrain on TRAIN+VALID and score on TEST
    X_trval_s = np.vstack([X_train_s, X_valid_s])
    y_trval   = np.concatenate([y_train, y_valid])
    final = RandomForestClassifier(**best_params).fit(X_trval_s, y_trval)
    prob_test = final.predict_proba(X_test_s)[:, 1]

    def pack(thr):
        pred = (prob_test >= thr).astype(int)
        return {
            "threshold": float(thr),
            "test_roc_auc": float(roc_auc_score(y_test, prob_test)),
            "test_average_precision": float(average_precision_score(y_test, prob_test)),
            "test_f1": float(f1_score(y_test, pred, zero_division=0)),
            "test_balanced_accuracy": float(balanced_accuracy_score(y_test, pred)),
            "test_accuracy": float(accuracy_score(y_test, pred)),
            "test_confusion_matrix": confusion_matrix(y_test, pred).tolist(),
            "test_classification_report": classification_report(y_test, pred, output_dict=True),
        }

    return {"F1_tuned": pack(thr_f1), "BalancedAcc_tuned": pack(thr_bal)}

print("Metric helpers ready.")


Metric helpers ready.


In [ ]:
# Seeds
seeds = derive_seeds_from_student_id(STUDENT_ID, n_seeds=5)
assert 0 <= SEED_INDEX < 5, "SEED_INDEX must be 0..4"
seed = seeds[SEED_INDEX]
np.random.seed(seed); random.seed(seed)
print("Derived seeds:", seeds)
print("Using ONLY seed:", seed)

# Data
raw = download_close_series(TICKER, START, END)
df2, y_full = derive_labels_from_close(raw)   # labels from Close abs log-returns (top 25%)
feats = build_close_only_features(df2)        # features (shifted by 1 day)

# Align labels to feature index, keep Date for splits
y = pd.Series(y_full, index=df2.index).reindex(feats.index).values.astype(int)
feats = feats.copy()
feats["Date"] = df2.loc[feats.index, "Date"].values

# Splits (chronological)
train_mask = feats["Date"] <= pd.to_datetime(TRAIN_END)
valid_mask = (feats["Date"] > pd.to_datetime(TRAIN_END)) & (feats["Date"] <= pd.to_datetime(VALID_END))
test_mask  = feats["Date"] > pd.to_datetime(VALID_END)

X_train = feats.loc[train_mask].drop(columns=["Date"]).values
y_train = y[train_mask.values]
X_valid = feats.loc[valid_mask].drop(columns=["Date"]).values
y_valid = y[valid_mask.values]
X_test  = feats.loc[test_mask].drop(columns=["Date"]).values
y_test  = y[test_mask.values]

print("Split sizes ->",
      "train:", X_train.shape[0],
      "valid:", X_valid.shape[0],
      "test:",  X_test.shape[0])

# Scaling: fit on TRAIN only
scaler = StandardScaler(with_mean=True, with_std=True)
X_train_s = scaler.fit_transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s  = scaler.transform(X_test)

print("Scaling complete.")


Derived seeds: [569498789, 554195645, 1795592910, 2769247983, 865482700]
Using ONLY seed: 569498789
Split sizes -> train: 1490 valid: 753 test: 396
Scaling complete.


In [ ]:
# ==== FAST Grid (sampling + low-cost proxy) ====
# Strategy:
# 1) During search, fix n_estimators to a small value (ESTIMATORS_FOR_SEARCH) to rank configs quickly.
# 2) Sample only SAMPLE_N random configs.
# 3) After picking the best (by VALID metric), do a tiny sweep of n_estimators on that winner to finalize.

SAMPLE_N = 64                 # try 32–128 depending on speed
ESTIMATORS_FOR_SEARCH = 200   # use 100–300 for fast ranking

param_grid = {
    "max_depth":           [6, 10, 14, 18],
    "min_samples_split":   [5, 10, 20, 30],
    "min_samples_leaf":    [3, 6, 10, 15],
    "max_features":        ["sqrt", "log2", None],
    "bootstrap":           [True, False],
}
fixed_params = {"class_weight": "balanced", "random_state": seed, "n_jobs": -1}

def sample_param_configs(grid: dict, n_samples: int, seed: int):
    rng = np.random.default_rng(seed)
    keys = list(grid.keys())
    choices = [grid[k] for k in keys]
    idxs = [rng.integers(0, len(c), size=n_samples) for c in choices]
    configs = []
    for i in range(n_samples):
        cfg = {}
        for k, arr, opts in zip(keys, idxs, choices):
            cfg[k] = opts[arr[i]]
        configs.append(cfg)
    return configs

# 1) Fast ranking phase (fixed small n_estimators)
grid_start = time.time()
grid_n = 0
grid_best_score = -1e9
grid_best = None

sampled_configs = sample_param_configs(param_grid, SAMPLE_N, seed)

for params in sampled_configs:
    grid_n += 1
    cfg = deepcopy(fixed_params)
    cfg.update(params)
    cfg["n_estimators"] = ESTIMATORS_FOR_SEARCH  # small for speed
    clf = RandomForestClassifier(**cfg)
    s = score_on_valid(clf, X_train_s, y_train, X_valid_s, y_valid, metric=METRIC)
    if s > grid_best_score:
        grid_best_score, grid_best = s, deepcopy(cfg)

# 2) Tiny sweep of n_estimators for the winner to finalize
final_sweep = [400, 800, 1200]  # you can trim to [600, 900] if needed
best_final = None
best_final_score = -1e9

for ne in final_sweep:
    cfg2 = deepcopy(grid_best)
    cfg2["n_estimators"] = ne
    clf2 = RandomForestClassifier(**cfg2)
    s2 = score_on_valid(clf2, X_train_s, y_train, X_valid_s, y_valid, metric=METRIC)
    if s2 > best_final_score:
        best_final_score, best_final = s2, deepcopy(cfg2)

grid_time = time.time() - grid_start
grid_best = best_final
grid_best_score = best_final_score

print(f"[GRID-FAST] Tried {grid_n} sampled configs (+ {len(final_sweep)} n_estimators sweep) in {grid_time:.1f}s")
print(f"[GRID-FAST] Best VALID {METRIC}: {grid_best_score:.4f}")
grid_best


[GRID-FAST] Tried 64 sampled configs (+ 3 n_estimators sweep) in 225.3s
[GRID-FAST] Best VALID roc_auc: 0.6288


{'class_weight': 'balanced',
 'random_state': 569498789,
 'n_jobs': -1,
 'max_depth': 6,
 'min_samples_split': 10,
 'min_samples_leaf': 15,
 'max_features': 'log2',
 'bootstrap': True,
 'n_estimators': 400}